# Equity forward curves

End-to-end tour of `market_structures/equity/`: term-structured dividend yields, discrete cash and proportional dividends, and bootstrapping a forward curve from market-quoted forwards. The final section plugs the resulting curve into an `InterpolatedVolSurface` to show the drop-in compatibility with the volatility stack.

## Environment setup

The cell below calls `setup_demo_env()` from `examples/_setup.py`, which performs three steps:

1. Adds the project root to `sys.path` so first-party packages (`database`, `schedules`, `market_structures`, `credit`, ...) import without installation.
2. Redirects the global SQLite path to `examples/demo.db` via `set_db_path()`. The production `quant.db` is never read or written by this notebook.
3. Creates the domain tables and seeds the holidays table (USD/EUR/GBP/PLN, years 2000-2100) on first run; subsequent runs detect existing data and skip seeding.

A status line is printed when the cell runs so you can see which path was taken.

In [ ]:
import sys
sys.path.insert(0, "..")

from examples._setup import setup_demo_env
setup_demo_env()

## Anchor: reference date, spot, and discount curve

Every curve below shares the same reference date (today's `t = 0`), the same spot, and the same risk-free `ZeroCurve`. We use a flat 4% continuously-compounded zero curve for clarity; in production any bootstrapped `ZeroCurve` works.

In [ ]:
from datetime import date, timedelta

import matplotlib.pyplot as plt
import numpy as np

from market_conventions import DayCountConvention
from market_structures.rates.curve import ZeroCurve

REF = date(2026, 1, 1)
SPOT = 100.0

pillars = [REF + timedelta(days=d) for d in (30, 180, 365, 730, 1825)]
zero_curve = ZeroCurve(
    reference_date=REF,
    pillar_dates=pillars,
    rates=[0.04] * len(pillars),
    day_count_convention=DayCountConvention.ACT_365_FIXED,
)

expiries = [REF + timedelta(days=d) for d in (30, 90, 180, 365, 540, 730, 1095, 1825)]
times = np.array([(e - REF).days / 365.0 for e in expiries])
times

## Curve 1 — flat dividend yield (parity with `EquityForward`)

`EquityForwardCurve.flat` produces a single-pillar curve numerically identical to the legacy `EquityForward(spot, zero_curve, q)`. This is the compatibility gate: every existing volatility-surface consumer using `EquityForward` keeps working when handed the new curve.

In [ ]:
from market_structures.equity import EquityForwardCurve
from market_structures.volatility.forward import EquityForward

q_flat = 0.02
legacy = EquityForward(SPOT, zero_curve, q_flat)
flat_curve = EquityForwardCurve.flat(SPOT, zero_curve, q_flat)

f_flat = np.array([flat_curve(e) for e in expiries])
f_legacy = np.array([legacy(e) for e in expiries])

print(f"max |diff| over expiries: {np.max(np.abs(f_flat - f_legacy)):.2e}")
print(f"F(1y) = S0 * exp((r-q)*T) = {SPOT * np.exp((0.04 - q_flat) * 1.0):.6f}")
print(f"flat_curve(1y)           = {flat_curve(1.0):.6f}")


## Curve 2 — term structure of dividend yields

Real names have a maturity-dependent dividend yield. `DividendYieldQuote` is a single point on that term structure; `EquityForwardCurve.from_dividend_yield_quotes` builds the curve.

Two interpolation modes are available:

- `FORWARD_YIELD_FLAT` (default) — cumulative yield `Q(T) = q(T) * T` is piecewise linear in `T`, equivalently the instantaneous forward dividend yield is piecewise constant on each segment. This is the equity analogue of the log-linear-in-DF convention used by the rates `ZeroCurve`.
- `LINEAR_IN_YIELD` — `q(T)` itself is piecewise linear between pillars.

Both modes agree at pillars and extrapolate flat in `q` outside the grid; they differ only in the inter-pillar shape.

In [ ]:
from market_structures.equity import (
    DividendYieldInterpolation,
    DividendYieldQuote,
)

div_quotes = [
    DividendYieldQuote(REF + timedelta(days=180), 0.010),
    DividendYieldQuote(REF + timedelta(days=365), 0.020),
    DividendYieldQuote(REF + timedelta(days=1825), 0.035),
]

ts_curve_fyf = EquityForwardCurve.from_dividend_yield_quotes(
    SPOT, zero_curve, div_quotes
)
ts_curve_liy = EquityForwardCurve.from_dividend_yield_quotes(
    SPOT, zero_curve, div_quotes,
    interpolation=DividendYieldInterpolation.LINEAR_IN_YIELD,
)

# Compare q(T) under both modes on a fine time grid.
t_grid = np.linspace(0.01, 6.0, 200)
q_fyf = np.array([ts_curve_fyf.dividend_yield(t) for t in t_grid])
q_liy = np.array([ts_curve_liy.dividend_yield(t) for t in t_grid])

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(t_grid, q_fyf, label="FORWARD_YIELD_FLAT (default)")
ax.plot(t_grid, q_liy, label="LINEAR_IN_YIELD", linestyle="--")
ax.scatter(ts_curve_fyf.pillar_times, ts_curve_fyf.pillar_yields, color="black", zorder=3, label="pillars")
ax.set_xlabel("T (years, ACT/365)")
ax.set_ylabel("q(T)")
ax.set_title("Dividend yield term structure — two interpolation modes")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## Curve 3 — discrete cash and proportional dividends

For single-stock names dividends are typically quoted as discrete amounts on ex-dates, not as a continuous yield. The Hull convention is

`F(T) = (S0 * Π_{ex_i <= T}(1 - p_i) - Σ_{ex_j <= T} d_j * DF(ex_j)) * exp(-q(T) * T) / DF(T)`

where `d_j` are cash dividends and `p_i` are proportional drops. `EquityForwardCurve.from_discrete_dividends` builds a curve directly from a list of `DiscreteDividend` instances and an optional flat `borrow_rate`. Note the *discontinuity* at every cash ex-date — `F(T)` jumps down by `d_j / DF(T)` as `T` crosses `ex_j`.

In [ ]:
from market_structures.equity import DiscreteDividend, DividendKind

# Two regular quarterly cash dividends and one 2% special proportional drop.
dividends = [
    DiscreteDividend(REF + timedelta(days=90),  0.60, DividendKind.CASH),
    DiscreteDividend(REF + timedelta(days=270), 0.65, DividendKind.CASH),
    DiscreteDividend(REF + timedelta(days=540), 0.02, DividendKind.PROPORTIONAL),
    DiscreteDividend(REF + timedelta(days=630), 0.70, DividendKind.CASH),
]

discrete_curve = EquityForwardCurve.from_discrete_dividends(
    SPOT, zero_curve, dividends, borrow_rate=0.0
)

# Sample F(T) on a fine date grid so we can see the jumps at ex-dates.
fine_days = np.arange(1, 1825)
fine_dates = [REF + timedelta(days=int(d)) for d in fine_days]
fine_times = fine_days / 365.0
f_discrete = np.array([discrete_curve(d) for d in fine_dates])

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(fine_times, f_discrete, label="discrete-dividend curve")
for d in dividends:
    t_ex = (d.ex_date - REF).days / 365.0
    ax.axvline(t_ex, linestyle=":", alpha=0.4, color="gray")
ax.set_xlabel("T (years, ACT/365)")
ax.set_ylabel("F(T)")
ax.set_title("Forward with discrete dividends (vertical lines = ex-dates)")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

## Curve 4 — bootstrapping from market-quoted forwards

When forwards (single-stock futures, EFPs, OTC forwards) are observable in the market, `EquityForwardCurveBootstrapper` inverts the textbook formula in closed form: `q_i = -log(F_i * DF(T_i) / S0) / T_i`. Mixed `ForwardQuote` / `DividendYieldQuote` input is accepted; both quote types pin one pillar each.

Below we sample forwards from the term-structured curve built earlier, hand them back as `ForwardQuote`s, and verify the round-trip recovers the original yields to machine precision.

In [ ]:
from market_structures.equity import (
    EquityForwardCurveBootstrapper,
    ForwardQuote,
)

sampled = [
    ForwardQuote(q.maturity_date, ts_curve_fyf.at_date(q.maturity_date))
    for q in div_quotes
]
for q in sampled:
    print(f"  ForwardQuote({q.maturity_date}, F={q.forward_price:.6f})")

bs = EquityForwardCurveBootstrapper(SPOT, zero_curve)
bootstrapped = bs.bootstrap(sampled)

print("\nRecovered yields vs original:")
for t_orig, q_orig, t_rec, q_rec in zip(
    ts_curve_fyf.pillar_times,
    ts_curve_fyf.pillar_yields,
    bootstrapped.pillar_times,
    bootstrapped.pillar_yields,
):
    print(f"  T={t_orig:.4f}: q_orig={q_orig:.10f}  q_recovered={q_rec:.10f}  diff={q_rec-q_orig:+.2e}")

## Using `EquityForwardCurve` as the `ForwardCallable` for a vol surface

The volatility-surface stack consumes its forward through the `ForwardCallable = Callable[[float], float]` alias declared in `market_structures.volatility.surface`. `EquityForwardCurve.__call__(when: float | date)` matches that signature, so any curve built above plugs directly into `InterpolatedVolSurface.__init__`, `SVISurface`, `SSVISurface`, or the calibration entry points without code changes.

In [ ]:
from market_structures.volatility import InterpolatedVolSurface

# Toy flat-vol grid: three expiries x three log-moneynesses.
surface_expiries = [0.25, 1.0, 2.0]
k_grid = [-0.20, 0.00, 0.20]
sigma = 0.22
total_variances = [[sigma**2 * t for _ in k_grid] for t in surface_expiries]

surface = InterpolatedVolSurface(
    reference_date=REF,
    forward=ts_curve_fyf,  # <-- the term-structured equity curve, used as a ForwardCallable
    expiries=surface_expiries,
    log_moneynesses=[k_grid] * len(surface_expiries),
    total_variances=total_variances,
)

# Sanity: implied_vol at the surface's own forward must return sigma at k=0.
F_1y = ts_curve_fyf(1.0)
print(f"Forward used by the surface: F(1y) = {F_1y:.6f}")
print(f"implied_vol(1y, K=F(1y))   = {surface.implied_vol(1.0, F_1y):.6f}")
print(f"expected (flat-vol grid)   = {sigma:.6f}")

## Summary

`EquityForwardCurve` is the term-structured equity counterpart of `ZeroCurve`: continuous-yield pillars (linear or forward-yield-flat interpolation), an optional discrete dividend schedule, and a closed-form bootstrapper from market `ForwardQuote`s. Because the curve duck-types as the volatility stack's `ForwardCallable`, every existing surface and calibration entry point consumes it without modification.